In [62]:
import pandas as pd
from sqlalchemy import create_engine, text

# Database credentials
username = 'admin'
password = 'admin%40123'
ip_address = '100.90.162.48'
port = '5432'
DB_URI = f'postgresql://{username}:{password}@{ip_address}:{port}'

db_name = "chicago_crime"

new_engine = create_engine(f'{DB_URI}/{db_name}')

In [63]:
from sqlalchemy.orm import sessionmaker

Session = sessionmaker(bind=new_engine)
session = Session()

In [64]:
with new_engine.connect() as conn:
    result = conn.execute(text("SELECT count(*) FROM chicago_crime where primary_type = 'THEFT'"))

print(result.fetchall())

[(307252,)]


In [65]:
df = pd.read_csv('Crimes_2020_to_Present_20260304.csv')

In [66]:
df.head()

,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,...,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,14121954,JK163066,02/23/2026 12:00:00 AM,069XX S CRANDON AVE,2825,OTHER OFFENSE,HARASSMENT BY TELEPHONE,APARTMENT,False,True,...,5.0,43.0,26,1192522.0,1859423.0,2026,03/02/2026 03:42:09 PM,41.769231,-87.569855,"(41.769231153, -87.569854828)"
1,14119135,JK159501,02/23/2026 12:00:00 AM,018XX N FAIRFIELD AVE,0710,THEFT,THEFT FROM MOTOR VEHICLE,STREET,False,False,...,1.0,22.0,06,1157750.0,1912034.0,2026,03/02/2026 03:42:09 PM,41.914378,-87.695882,"(41.914378059, -87.695881521)"
2,14119149,JK159544,02/23/2026 12:00:00 AM,016XX W MORSE AVE,1754,OFFENSE INVOLVING CHILDREN,AGGRAVATED SEXUAL ASSAULT OF CHILD BY FAMILY M...,OTHER (SPECIFY),False,True,...,49.0,1.0,02,1164302.0,1946138.0,2026,03/02/2026 03:42:09 PM,42.007825,-87.670842,"(42.007824765, -87.670841822)"
3,14124471,JK163214,02/23/2026 12:00:00 AM,018XX S AVERS AVE,1750,OFFENSE INVOLVING CHILDREN,CHILD ABUSE,APARTMENT,False,True,...,24.0,29.0,08B,NaN,NaN,2026,03/02/2026 03:42:09 PM,NaN,NaN,NaN
4,14119441,JK159902,02/23/2026 12:00:00 AM,063XX W CUYLER AVE,0810,THEFT,OVER $500,STREET,False,False,...,38.0,15.0,06,1133380.0,1926251.0,2026,03/02/2026 03:42:09 PM,41.953853,-87.785080,"(41.953853257, -87.785080407)"


## SQL Agent

### Model

In [96]:
OLLAMA_MODEL   = "mannix/defog-llama3-sqlcoder-8b"                        # Ollama model tag
OLLAMA_BASE_URL = "http://localhost:11434"     # default Ollama URL 100.90.162.48

from langchain_ollama import ChatOllama

model = ChatOllama(
    model=OLLAMA_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0,              # deterministic SQL generation
)

### Connecting to SQL Database

In [97]:
from langchain_community.utilities import SQLDatabase

# ── Connection settings (from test.ipynb) ─────────────────────
DB_USER     = "admin"
DB_PASSWORD = "admin%40123"          # URL-encoded: @ → %40
DB_HOST     = "100.90.162.48"
DB_PORT     = "5432"
DB_NAME     = "chicago_crime"

# Build a standard SQLAlchemy URI
DB_URI = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Wrap it with LangChain's SQLDatabase so the agent can use it
# include_tables=None means all tables are accessible;
# pass a list like ['chicago_crime'] to restrict access.
db = SQLDatabase.from_uri(DB_URI)

print(f"Dialect         : {db.dialect}")
print(f"Available tables: {db.get_usable_table_names()}")

Dialect         : postgresql
Available tables: ['chicago_crime']


### Creating SQL Tools

In [98]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

# Create the toolkit — it needs both the DB wrapper and the LLM
# (the LLM is used internally by the query-checker tool)
toolkit = SQLDatabaseToolkit(db=db, llm=model)
tools   = toolkit.get_tools()

# Quick sanity check: print all available tools and their descriptions
for t in tools:
    print(f"{t.name}:\n  {t.description[:120]}\n")

sql_db_query:
  Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correc

sql_db_schema:
  Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure t

sql_db_list_tables:
  Input is an empty string, output is a comma-separated list of tables in the database.

sql_db_query_checker:
  Use this tool to double check if your query is correct before executing it. Always use this tool before executing a quer



### Build Agent

All agent logic lives in `sql_agent.py`:
- `build_agent(model, db, tools)` — wires the LangGraph nodes and returns a compiled agent
- `run_agent(agent, question)` — streams the agent and prints each step with SQL highlighted

The graph: `list_tables → call_get_schema → get_schema → generate_query → check_query → run_query`

In [99]:
# All agent logic (nodes, graph wiring, fallback helpers) lives in sql_agent.py
import importlib, sql_agent
importlib.reload(sql_agent)                   # pick up any changes to sql_agent.py
from sql_agent import build_agent, run_agent

# Compile the agent using the model, DB wrapper, and tools set up in the cells above
agent = build_agent(model, db, tools)
print("Agent compiled successfully!")

Agent compiled successfully!


### Run the Agent

Pass any natural-language question. Generated SQL and query results are highlighted in the output.

In [101]:
run_agent(agent, "What are the Top 3 crimes in 2025 along with the counts?")


Question: What are the Top 3 crimes in 2025 along with the counts?
================================ Human Message =================================

What are the Top 3 crimes in 2025 along with the counts?
================================== Ai Message ==================================

Available tables: chicago_crime


ResponseError: registry.ollama.ai/mannix/defog-llama3-sqlcoder-8b:latest does not support tools (status code: 400)

In [ ]:
run_agent(agent, "In 2025, which month had the highest number of THEFT crimes?")


Question: In 2024, which month had the highest number of THEFT crimes?
================================ Human Message =================================

In 2024, which month had the highest number of THEFT crimes?
================================== Ai Message ==================================

Available tables: chicago_crime
================================== Ai Message ==================================



{"name": "sql_db_query", "parameters": {"query": "SELECT month FROM chicago_crime WHERE year = 2024 AND crime_type = 'THEFT' GROUP BY month ORDER BY COUNT(*) DESC LIMIT 1", "db": "chicago_crime"}}
Tool Calls:
  sql_db_schema (fallback_schema)
 Call ID: fallback_schema
  Args:
    table_names: chicago_crime
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE chicago_crime (
	"ID" BIGINT, 
	case_number TEXT, 
	"Date" TEXT, 
	"Block" TEXT, 
	"IUCR" TEXT, 
	crime_type TEXT, 
	"Description" TEXT, 
	location_description TEX

In [72]:
# Quick verification run — compact output to confirm no KeyError and SQL is generated
errors = []
sqls_seen = []
results_seen = []

for step_no, step in enumerate(
    agent.stream(
        {"messages": [{"role": "user", "content": "What are the Top 3 crime types in 2025?"}]},
        stream_mode="values",
    ),
    start=1,
):
    msg = step["messages"][-1]
    for tc in getattr(msg, "tool_calls", None) or []:
        if tc.get("name") == "sql_db_query":
            sqls_seen.append(tc.get("args", {}).get("query", ""))
    if msg.__class__.__name__ == "ToolMessage" and getattr(msg, "name", "") == "sql_db_query":
        results_seen.append(msg.content[:200])

print(f"SQL queries generated : {len(sqls_seen)}")
for i, s in enumerate(sqls_seen, 1):
    print(f"  SQL {i}: {s[:120]}")
print(f"\nDB results received   : {len(results_seen)}")
for i, r in enumerate(results_seen, 1):
    print(f"  Result {i}: {r}")
print("\nNo KeyError raised ✓" if not errors else errors)

SQL queries generated : 2
  SQL 1: SELECT "primary_type" FROM chicago_crime WHERE "Year" = 2025 GROUP BY "primary_type" ORDER BY COUNT(*) DESC LIMIT 3
  SQL 2: SELECT "primary_type" FROM chicago_crime WHERE "Year" = 2025 GROUP BY "primary_type" ORDER BY COUNT(*) DESC LIMIT 3

DB results received   : 1
  Result 1: [('THEFT',), ('BATTERY',), ('CRIMINAL DAMAGE',)]

No KeyError raised ✓
